In [5]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import Trainer  # Add this line
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
import numpy as np

class CryptoDataset(Dataset):
    def __init__(self, features: pd.DataFrame, labels: pd.DataFrame, transform=None):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(LabelEncoder().fit_transform(labels['&-target']), dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = self.features[idx]
        if self.transform:
            feature = self.transform(feature)
        return feature, self.labels[idx]

class CryptoDataModule(pl.LightningDataModule):
    def __init__(self, directory_path, batch_size=128, num_workers=4, train_split=0.8):
        super().__init__()
        self.directory_path = directory_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_split = train_split
        self.scaler = StandardScaler()
        
    def setup(self, stage=None):
        # Load and preprocess data with new file paths
        features_path = os.path.join(self.directory_path, 'ETH/USDT:USDT_features_20250203_114859.parquet')
        labels_path = os.path.join(self.directory_path, 'ETH/USDT:USDT_labels_20250203_114859.parquet')
        
        if not (os.path.exists(features_path) and os.path.exists(labels_path)):
            raise FileNotFoundError(f"Data files not found in {self.directory_path}/ETH/")
            
        features = pd.read_parquet(features_path)
        labels = pd.read_parquet(labels_path)
        
        # Scale features
        features_scaled = self.scaler.fit_transform(features)
        features = pd.DataFrame(features_scaled, columns=features.columns)
        
        # Create dataset
        dataset = CryptoDataset(features, labels)
        
        # Split dataset
        train_size = int(self.train_split * len(dataset))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        
        # Store dimensions
        self.input_dim = features.shape[1]
        self.output_dim = len(labels['&-target'].unique())

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, 
                        num_workers=self.num_workers, shuffle=True, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, 
                        num_workers=self.num_workers, pin_memory=True)


In [1]:
# Update CryptoPricePredictor class
class CryptoPricePredictor(pl.LightningModule):
    def __init__(self, input_dim, hidden_dims=[512, 256, 128], dropout_rate=0.3, learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()
        
        # Calculate class weights for imbalanced dataset
        self.register_buffer('class_weights', None)  # Will be set in training_step
        
        # Model architecture with smaller last hidden layer
        layers = []
        dims = [input_dim] + hidden_dims
        
        for i in range(len(dims)-1):
            layers.extend([
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.LeakyReLU(0.1),  # Changed to LeakyReLU
                nn.Dropout(dropout_rate)
            ])
        
        # Smaller final layer for better focus
        layers.append(nn.Linear(dims[-1], 2))
        self.model = nn.Sequential(*layers)
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def _calculate_win_metrics(self, preds, labels):
        # Calculate WIN class precision (class 1)
        win_predictions = (preds == 1)
        true_wins = (labels == 1)
        
        true_positives = (win_predictions & true_wins).float().sum()
        false_positives = (win_predictions & ~true_wins).float().sum()
        
        win_precision = true_positives / (true_positives + false_positives + 1e-8)
        win_recall = true_positives / (true_wins.float().sum() + 1e-8)
        
        return {
            'win_precision': win_precision,
            'win_recall': win_recall,
            'win_predictions': win_predictions.float().mean(),  # Proportion of WIN predictions
            'true_wins': true_wins.float().mean()  # Actual WIN ratio
        }

    def training_step(self, batch, batch_idx):
        x, y = batch
        
        # Compute class weights if not already done
        if self.class_weights is None:
            total = y.size(0)
            win_count = (y == 1).sum().float()
            lose_count = (y == 0).sum().float()
            self.class_weights = torch.tensor([total/(2*lose_count), total/(2*win_count)])
            
        logits = self(x)
        # Use weighted loss
        loss = F.cross_entropy(logits, y, weight=self.class_weights.to(y.device))
        
        # Add confidence threshold for predictions
        probabilities = F.softmax(logits, dim=1)
        confident_predictions = (probabilities > 0.6).float()  # Only predict when confidence > 60%
        preds = torch.argmax(confident_predictions, dim=1)
        
        metrics = self._calculate_win_metrics(preds, y)
        
        self.log_dict({
            'train_loss': loss,
            'train_win_precision': metrics['win_precision'],
            'train_win_recall': metrics['win_recall'],
            'train_win_pred_ratio': metrics['win_predictions'],
            'train_win_true_ratio': metrics['true_wins']
        }, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y, weight=self.class_weights.to(y.device))
        
        # Use same confidence threshold
        probabilities = F.softmax(logits, dim=1)
        confident_predictions = (probabilities > 0.6).float()
        preds = torch.argmax(confident_predictions, dim=1)
        
        metrics = self._calculate_win_metrics(preds, y)
        
        self.log_dict({
            'val_loss': loss,
            'val_win_precision': metrics['win_precision'],
            'val_win_recall': metrics['win_recall'],
            'val_win_pred_ratio': metrics['win_predictions'],
            'val_win_true_ratio': metrics['true_wins']
        }, prog_bar=True)
        
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_win_precision"  # Changed to monitor WIN precision
            }
        }


NameError: name 'pl' is not defined

In [6]:
if __name__ == "__main__":
    # Directory and file setup
    directory_path = '/allah/data/parquet'
    os.chdir(directory_path)
    
    # Initialize and setup data module
    data_module = CryptoDataModule(directory_path=directory_path, batch_size=64)
    data_module.setup()

    # Print initial WIN ratio in dataset
    train_labels = torch.cat([y for _, y in data_module.train_dataloader()])
    win_ratio = (train_labels == 1).float().mean()
    print(f"Initial WIN ratio in training set: {win_ratio:.2%}")

    # Model initialization
    model = CryptoPricePredictor(input_dim=data_module.input_dim)

    # Training setup with WIN precision focus
    checkpoint_callback = ModelCheckpoint(
        monitor="val_win_precision",
        mode="max",
        filename="best-win-precision-{epoch:02d}-{val_win_precision:.2f}",
        save_top_k=3,
        verbose=True
    )
    
    early_stopping = EarlyStopping(
        monitor="val_win_precision",
        mode="max",
        patience=10,
        verbose=True
    )

    logger = TensorBoardLogger(save_dir=directory_path, name="win_precision_logs")

    # Initialize trainer
    trainer = Trainer(
        max_epochs=1000,
        callbacks=[early_stopping, checkpoint_callback],
        logger=logger,
        accelerator='auto',
        devices=1
    )

    # Train the model
    trainer.fit(model, datamodule=data_module)

    # Print best WIN precision achieved
    print(f"Best WIN precision: {checkpoint_callback.best_model_score:.4f}")


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Initial WIN ratio in training set: 36.81%


AttributeError: module 'torch.optim' has no attribute 'ReduceLROnPlateau'